In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini 3.5 Live Translate

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/gemini_3_5_live_translate.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Faudio%2Fspeech%2Fgetting-started%2Fgemini_3_5_live_translate.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/audio/speech/getting-started/gemini_3_5_live_translate.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/gemini_3_5_live_translate.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/gemini_3_5_live_translate.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/gemini_3_5_live_translate.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/gemini_3_5_live_translate.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/gemini_3_5_live_translate.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/gemini_3_5_live_translate.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Author |
| --- |
| [Katie Nguyen](https://github.com/katiemn) |

## Overview

This notebook introduces Gemini 3.5 Translate, Google's model for low-latency, real-time speech-to-speech translation between 70+ languages, available through Agent Platform.

In this tutorial, you'll learn how to use Gemini 3.5 Translate with the Google Gen AI SDK, through the streaming `BidiGenerateContent` (Live) API, to:

- Generate translated audio and text from an audio file
- Translate long audio files
- Stream translation from a microphone

## Get started

### Install Google Gen AI SDK for Python & other libraries

Install the Google Gen AI SDK along with a few supporting libraries used later in this notebook.

In [ ]:
%pip install --upgrade --quiet google-genai ipywebrtc numpy pydub > /dev/null 2>&1

### Authenticate your notebook environment (Colab only)

If you are running this notebook on Google Colab, run the following cell to authenticate your environment.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Import libraries

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
import asyncio
import os
import numpy as np

from IPython.display import Audio, display, Markdown
from google import genai
from google.genai import types
from pydub import AudioSegment
from ipywebrtc import AudioRecorder, CameraStream

### Set Google Cloud project information

To get started using Agent Platform, you must have an existing Google Cloud project and [enable the Agent Platform API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project](https://docs.cloud.google.com/resource-manager/docs/creating-managing-projects) and a [development environment](https://cloud.google.com/docs/authentication/set-up-adc-local-dev-environment).

In [ ]:
# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "global")

client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)

### Load the Gemini 3.5 Translate model

In [ ]:
MODEL_ID = "gemini-3.5-live-translate-preview"

## Translate streaming audio

**Streaming:** A live session stays open while you stream small chunks of audio to the model and receive translated results incrementally, as they become available. This is the approach you'd use for near real-time captioning or translating live audio, such as microphone input.

### Define helper functions for streaming translation

The following helper functions manage a streaming translation session and are reused throughout the rest of this notebook:

- `play_audio(audio_bytes, rate)`: Decodes raw 16-bit PCM bytes and renders an inline audio player for the translated audio.
- `send_streaming_audio(session, audio_file)`: Loads an audio file, converts it to 16 kHz mono 16-bit PCM, and streams it to the live session in small (100 ms) chunks in real time. It appends a short tail of silence to flush the final results, then signals the end of the audio stream.
- `receive_streaming_messages(session, interim_display, transcript_buffer, translation_buffer, audio_chunks)`: Consumes server messages as they arrive, accumulating the source transcription, the translated text, and the translated audio, while refreshing the live display as new text arrives.
- `streaming_main(audio_file, config, interim_display)`: Opens a live session, runs the send and receive tasks concurrently, and then plays the translated audio and renders the final translation alongside the source text.

In [ ]:
SEND_SAMPLE_RATE = 16000
OUTPUT_SAMPLE_RATE = 24000

def play_audio(audio_bytes, rate=OUTPUT_SAMPLE_RATE):
    audio_array = np.frombuffer(audio_bytes, dtype="<i2")
    display(Audio(data=audio_array, rate=rate))


async def send_streaming_audio(session, audio_file):
    try:
        seg = AudioSegment.from_file(audio_file)
        seg = seg.set_channels(1).set_frame_rate(SEND_SAMPLE_RATE).set_sample_width(2)
        samples = np.array(seg.get_array_of_samples(), dtype="<i2")

        chunk_duration = 0.1
        chunk_frames = int(SEND_SAMPLE_RATE * chunk_duration)
        pacing_delay = chunk_duration
        mime_type = f"audio/pcm;rate={SEND_SAMPLE_RATE}"

        for start in range(0, len(samples), chunk_frames):
            block = samples[start:start + chunk_frames]
            await session.send_realtime_input(audio=types.Blob(data=block.tobytes(), mime_type=mime_type))
            await asyncio.sleep(pacing_delay)

        silence = np.zeros(chunk_frames, dtype="<i2").tobytes()
        for _ in range(int(2.0 / chunk_duration)):
            await session.send_realtime_input(
                audio=types.Blob(data=silence, mime_type=mime_type)
            )
            await asyncio.sleep(pacing_delay)
    except Exception as e:
        print(f"Error reading/streaming wav file: {e}")
    finally:
        await session.send_realtime_input(audio_stream_end=True)


async def receive_streaming_messages(session, interim_display, transcript_buffer, translation_buffer, audio_chunks):
    def render():
        interim_display.update(Markdown(f"**Translation:** {' '.join(translation_buffer)}"))
    try:
        async for message in session.receive():
            if not message.server_content:
                continue
            server_content = message.server_content

            final = server_content.input_transcription
            if final and final.text:
                transcript_buffer.append(final.text)
                render()
            out = server_content.output_transcription
            if out and out.text:
                translation_buffer.append(out.text)
                render()
            if server_content.model_turn:
                for part in server_content.model_turn.parts:
                    if part.inline_data and part.inline_data.mime_type.startswith("audio"):
                        audio_chunks.append(part.inline_data.data)
    except Exception as e:
        print(f"Error translating file: {e}")


async def streaming_main(audio_file, config, interim_display):
    transcript_buffer, translation_buffer, audio_chunks = [], [], []

    async with client.aio.live.connect(model=MODEL_ID, config=config) as session:
        if session.setup_complete is None:
            print("No setup_complete received from server.")
            return
        send_task = asyncio.create_task(send_streaming_audio(session, audio_file))
        recv_task = asyncio.create_task(receive_streaming_messages(session, interim_display, transcript_buffer, translation_buffer, audio_chunks))

        await send_task
        try:
            await asyncio.wait_for(recv_task, timeout=2.0)
        except asyncio.TimeoutError:
            recv_task.cancel()

    if audio_chunks:
        play_audio(b"".join(audio_chunks))
    else:
        print("No translated audio received.")
    interim_display.update(Markdown(
        f"**Translation:** {' '.join(translation_buffer).strip()}\n\n"
        f"**Source:** {' '.join(transcript_buffer).strip()}"
    ))


### Translation

To translate streaming audio, build a `LiveConnectConfig` with the following fields:

- `response_modalities`: The output types to return. Here, both translated `AUDIO` and `TEXT` are requested.
- `input_audio_transcription`: Enables a text transcription of the source (input) audio.
- `output_audio_transcription`: Enables a text transcription of the translated (output) audio.
- `translation_config`: Configures the translation. 
    - `target_language_code` sets the language to translate into (for example, `"ja"` for Japanese). See supported languages and codes in the [documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/gemini/3-5-live-translate#supported-languages).
    - `echo_target_language` If `True`, the model echoes back speech that is already in the target language. If `False`, the model will stay silent when the input speech is already in the target language.

Run the following cell to download and play the audio you'll be streaming, then run the cell after it to start the streaming session.

In [ ]:
audio_file_url = "https://storage.googleapis.com/cloud-samples-data/generative-ai/audio/tell-a-story.wav"
audio_file = "input.wav"

!wget -q $audio_file_url -O "input.wav"
display(Audio(filename=audio_file, autoplay=False))

In [ ]:
config = types.LiveConnectConfig(
    response_modalities=["AUDIO", "TEXT"],
    input_audio_transcription=types.AudioTranscriptionConfig(),
    output_audio_transcription=types.AudioTranscriptionConfig(),
    translation_config=types.TranslationConfig(
        target_language_code="ja",
        echo_target_language=False
    )
)
interim_display = display(Markdown(""), display_id=True)
await streaming_main(audio_file, config, interim_display)

### Translation with multiple languages

The model automatically detects the source language, so a single session can translate audio that switches between languages. The following clip contains both Korean and English speech, and the configuration below translates all of it into French.

Once again, run the following cell to download and play the audio you'll be translating.

In [ ]:
audio_file_url = "https://storage.googleapis.com/cloud-samples-data/generative-ai/audio/korean-english.wav"
audio_file = "input.wav"

!wget -q $audio_file_url -O "input.wav"
display(Audio(filename=audio_file, autoplay=False))

In [ ]:
config = types.LiveConnectConfig(
    response_modalities=["AUDIO", "TEXT"],
    input_audio_transcription=types.AudioTranscriptionConfig(),
    output_audio_transcription=types.AudioTranscriptionConfig(),
    translation_config=types.TranslationConfig(
        target_language_code="fr",
    )
)
interim_display = display(Markdown(""), display_id=True)
await streaming_main(audio_file, config, interim_display)

### Translating long audio with session chunking

Streaming sessions have a practical length limit. For longer recordings, split the audio into overlapping chunks and open a new Live session for each one. Run the following cell to download a longer audio file.

In [ ]:
audio_file_url = "https://storage.googleapis.com/cloud-samples-data/generative-ai/audio/Accessible_writing_tip_Informative_semantic_titles_and_headings.mp3"
audio_file = "input.wav"

!wget -q $audio_file_url -O "input.wav"
display(Audio(filename=audio_file, autoplay=False))

In [ ]:
config = types.LiveConnectConfig(
    response_modalities=["AUDIO", "TEXT"],
    input_audio_transcription=types.AudioTranscriptionConfig(),
    output_audio_transcription=types.AudioTranscriptionConfig(),
    translation_config=types.TranslationConfig(
        target_language_code="es",
    )
)


async def translate_long_audio(audio_file, config):
    print(f"Loading '{audio_file}' into memory...")
    audio = AudioSegment.from_file(audio_file)

    chunk_length_ms = 150000
    overlap_ms = 500

    temp_file_path = "temp_chunk.wav"

    try:
        for i, chunk_start in enumerate(range(0, len(audio), chunk_length_ms)):
            # Handle boundary overlap
            actual_start = max(0, chunk_start - overlap_ms) if i > 0 else chunk_start
            chunk_end = min(chunk_start + chunk_length_ms, len(audio))

            # Slices and exports the chunk to wav
            chunk = audio[actual_start:chunk_end]
            chunk.export(temp_file_path, format="wav")

            interim_display = display(
                Markdown(f"**Chunk {i+1}:** *Initializing...*"),
                display_id=True
            )

            await streaming_main(
                temp_file_path,
                config,
                interim_display,
            )

    finally:
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)
            print("\nCleaned up temporary chunk file.")


await translate_long_audio(audio_file, config)

### Translation from microphone input

In the following cells, you'll simulate translating from an audio stream. To start, you'll record an audio clip with your microphone by running the following cell. You might need to run it twice after granting microphone permission.

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import output

    output.enable_custom_widget_manager()

camera = CameraStream(constraints={"audio": True, "video": False})
recorder = AudioRecorder(stream=camera)
recorder

Once the audio is captured and you've stopped recording, you'll use FFmpeg to convert and save the clip to an MP3 file for processing. (FFmpeg comes preinstalled in Colab; install it locally if you're running this notebook elsewhere.)

In [ ]:
with open("recording.webm", "wb") as f:
    f.write(recorder.audio.value)

audio_file = "recording.mp3"
!ffmpeg -i recording.webm -vn -ar 16000 -ac 1 -f mp3 recording.mp3

Now, you'll read the audio file and generate audio chunks to simulate streaming using the previously defined helper functions.

In [ ]:
config = types.LiveConnectConfig(
    response_modalities=["AUDIO", "TEXT"],
    input_audio_transcription=types.AudioTranscriptionConfig(),
    output_audio_transcription=types.AudioTranscriptionConfig(),
    translation_config=types.TranslationConfig(
        target_language_code="vi",
    )
)
interim_display = display(Markdown(""), display_id=True)
await streaming_main(audio_file, config, interim_display)